In [6]:
import pandas as pd
df = pd.read_csv("../data/raw/epiclim_raw.csv")
df.head()


,Unnamed: 0,week_of_outbreak,state_ut,district,Disease,Cases,Deaths,day,mon,year,Latitude,Longitude,preci,LAI,Temp
0,0,1st week,Meghalaya,East Jaintia Hills,Acute Diarrhoeal Disease,160,NaN,2,1,2022,25.251576,92.484050,0.020354,34.5,291.533333
1,1,2nd week,Maharashtra,Gadchiroli,Malaria,7,2.0,10,1,2022,19.759070,80.162281,0.007479,9.0,299.970000
2,2,3rd week,Tamil Nadu,Pudukottai,Acute Diarrhoeal Disease,8,NaN,18,1,2022,10.382651,78.819126,0.107413,12.0,300.766667
3,3,3rd week,Gujarat,Patan,Acute Diarrhoeal Disease,7,NaN,11,1,2022,23.774057,71.683735,0.065094,9.0,299.080000
4,4,3rd week,Kerala,Ernakulam,Acute Diarrhoeal Disease,14,NaN,24,12,2021,9.984080,76.274146,0.041256,33.0,303.028000


In [7]:
df.shape


(8985, 15)

In [9]:
df_dengue = df[df['Disease'].str.lower() == 'dengue']
df_dengue.head()

,Unnamed: 0,week_of_outbreak,state_ut,district,Disease,Cases,Deaths,day,mon,year,Latitude,Longitude,preci,LAI,Temp
23,23,10th week,Telangana,Nalgonda,Dengue,6,NaN,11,3,2022,16.857964,79.217494,0.000134,8.0,309.556
25,25,10th week,Kerala,Trivandrum,Dengue,9,NaN,22,2,2022,8.488227,76.947551,0.093704,46.0,301.496
26,26,10th week,Tamil Nadu,Ariyalur,Dengue,6,NaN,5,2,2022,11.076036,79.117455,0.012509,10.0,303.384
28,28,10th week,Tamil Nadu,Virudhunagar,Dengue,17,NaN,16,2,2022,9.520894,77.878456,0.307975,7.0,306.000
37,37,12th week,Maharashtra,Raigad,Dengue,11,NaN,21,3,2022,18.492809,73.138071,0.000180,NaN,312.184


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8985 entries, 0 to 8984
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        8985 non-null   int64  
 1   week_of_outbreak  8985 non-null   object 
 2   state_ut          8985 non-null   object 
 3   district          8985 non-null   object 
 4   Disease           8985 non-null   object 
 5   Cases             8985 non-null   object 
 6   Deaths            2554 non-null   float64
 7   day               8985 non-null   int64  
 8   mon               8985 non-null   int64  
 9   year              8985 non-null   int64  
 10  Latitude          8985 non-null   float64
 11  Longitude         8985 non-null   float64
 12  preci             8849 non-null   float64
 13  LAI               6790 non-null   float64
 14  Temp              8047 non-null   float64
dtypes: float64(6), int64(4), object(5)
memory usage: 1.0+ MB


In [11]:
# Check missing values
df.isnull().sum()

Unnamed: 0             0
week_of_outbreak       0
state_ut               0
district               0
Disease                0
Cases                  0
Deaths              6431
day                    0
mon                    0
year                   0
Latitude               0
Longitude              0
preci                136
LAI                 2195
Temp                 938
dtype: int64

In [13]:
# Fill missing numerical values with 0
num_cols = ['Cases', 'Deaths', 'preci', 'LAI', 'Temp']
df[num_cols] = df[num_cols].fillna(0)

# Drop rows with missing critical location info
df = df.dropna(subset=['state_ut', 'district', 'Latitude', 'Longitude'])

In [14]:
# Create a date column
df['date'] = pd.to_datetime(
    dict(year=df['year'], month=df['mon'], day=df['day']),
    errors='coerce'
)

# Sort data for time-series operations
df = df.sort_values(by=['state_ut', 'district', 'date'])

In [15]:
df.head()


,Unnamed: 0,week_of_outbreak,state_ut,district,Disease,Cases,Deaths,day,mon,year,Latitude,Longitude,preci,LAI,Temp,date
837,837,23rd week,Andaman and Nicobar Islands,Andaman,Acute Diarrhoeal Disease,8,0.0,6,6,2019,13.511120,92.917388,0.772605,33.0,301.720000,2019-06-06
1192,1192,41st week,Andaman and Nicobar Islands,Andaman,Acute Diarrhoeal Disease,30,0.0,9,10,2019,13.511120,92.917388,0.149858,42.0,298.906667,2019-10-09
6193,6193,51st week,Andaman and Nicobar Islands,North and Middle Andaman,Acute Diarrhoeal Disease,86,0.0,16,12,2013,12.611239,92.831654,0.188149,53.0,298.200000,2013-12-16
8192,8192,24th week,Andhra Pradesh,Anantapur,Malaria,68,0.0,9,6,2010,14.654623,77.556260,0.254048,0.0,311.175000,2010-06-09
6526,6526,25th week,Andhra Pradesh,Anantapur,Acute Diarrhoeal Disease,23,0.0,26,6,2011,14.654623,77.556260,0.027715,0.0,307.820000,2011-06-26


In [16]:
df.isnull().sum()


Unnamed: 0          0
week_of_outbreak    0
state_ut            0
district            0
Disease             0
Cases               0
Deaths              0
day                 0
mon                 0
year                0
Latitude            0
Longitude           0
preci               0
LAI                 0
Temp                0
date                0
dtype: int64

In [17]:
df.shape


(8985, 16)

In [18]:
df[['year', 'mon', 'day', 'date']].head(10)


,year,mon,day,date
837,2019,6,6,2019-06-06
1192,2019,10,9,2019-10-09
6193,2013,12,16,2013-12-16
8192,2010,6,9,2010-06-09
6526,2011,6,26,2011-06-26
7342,2011,6,26,2011-06-26
7383,2011,6,26,2011-06-26
7638,2011,9,3,2011-09-03
6422,2012,5,18,2012-05-18
7026,2012,12,20,2012-12-20


In [19]:
# Create year-month column
df['year_month'] = df['date'].dt.to_period('M')

# Monthly aggregation
df_monthly = df.groupby(
    ['state_ut', 'district', 'year_month', 'Latitude', 'Longitude'],
    as_index=False
).agg({
    'Cases': 'sum',
    'Deaths': 'sum',
    'preci': 'mean',
    'LAI': 'mean',
    'Temp': 'mean'
})

df_monthly.head()


,state_ut,district,year_month,Latitude,Longitude,Cases,Deaths,preci,LAI,Temp
0,Andaman and Nicobar Islands,Andaman,2019-06,13.511120,92.917388,8,0.0,0.772605,33.0,301.720000
1,Andaman and Nicobar Islands,Andaman,2019-10,13.511120,92.917388,30,0.0,0.149858,42.0,298.906667
2,Andaman and Nicobar Islands,North and Middle Andaman,2013-12,12.611239,92.831654,86,0.0,0.188149,53.0,298.200000
3,Andhra Pradesh,Anantapur,2010-06,14.654623,77.556260,68,0.0,0.254048,0.0,311.175000
4,Andhra Pradesh,Anantapur,2011-06,14.654623,77.556260,232382,0.0,0.027715,0.0,307.820000


In [21]:
df_monthly['Cases'].dtype


dtype('O')

In [22]:

df_monthly['Cases'] = pd.to_numeric(df_monthly['Cases'], errors='coerce')

df_monthly['Cases'] = df_monthly['Cases'].fillna(0)


In [23]:
df_monthly['Cases'].dtype


dtype('float64')

In [24]:
df_monthly = df_monthly.sort_values(
    by=['state_ut', 'district', 'year_month']
)


In [25]:
df_monthly['Cases_MA_3'] = df_monthly.groupby(
    ['state_ut', 'district']
)['Cases'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)

In [26]:
df_monthly[['Cases', 'Cases_MA_3']].head()


,Cases,Cases_MA_3
0,8.0,8.0
1,30.0,19.0
2,86.0,86.0
3,68.0,68.0
4,232382.0,116225.0


In [27]:
df_monthly['Deaths'] = pd.to_numeric(df_monthly['Deaths'], errors='coerce').fillna(0)

df_monthly['Deaths_MA_3'] = df_monthly.groupby(
    ['state_ut', 'district']
)['Deaths'].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)


In [28]:
df_monthly['Case_Growth_Rate'] = df_monthly.groupby(
    ['state_ut', 'district']
)['Cases'].pct_change().fillna(0)


In [29]:
df_monthly['Month'] = df_monthly['year_month'].dt.month

df_monthly['Season'] = df_monthly['Month'].apply(
    lambda x: 'Winter' if x in [12,1,2]
    else 'Summer' if x in [3,4,5]
    else 'Monsoon' if x in [6,7,8,9]
    else 'Post-Monsoon'
)


In [32]:
df_monthly.columns


Index(['state_ut', 'district', 'year_month', 'Latitude', 'Longitude', 'Cases',
       'Deaths', 'preci', 'LAI', 'Temp', 'Cases_MA_3', 'Deaths_MA_3',
       'Case_Growth_Rate', 'Month', 'Season'],
      dtype='object')

In [38]:
df_monthly['Cases_MA_3']


0            8.000000
1           19.000000
2           86.000000
3           68.000000
4       116225.000000
            ...      
6704    273002.666667
6705    273004.000000
6706    272971.333333
6707        70.000000
6708        48.000000
Name: Cases_MA_3, Length: 6709, dtype: float64

In [39]:
df_monthly['Case_Growth_Rate']


0          0.000000
1          2.750000
2          0.000000
3          0.000000
4       3416.382353
           ...     
6704    5646.062069
6705      -0.999947
6706       0.093023
6707       0.000000
6708      -0.628571
Name: Case_Growth_Rate, Length: 6709, dtype: float64

In [40]:
df_monthly['Season']

0            Monsoon
1       Post-Monsoon
2             Winter
3            Monsoon
4            Monsoon
            ...     
6704         Monsoon
6705         Monsoon
6706         Monsoon
6707         Monsoon
6708         Monsoon
Name: Season, Length: 6709, dtype: object